# 形態素解析・サブワード分割と固有表現認識（系列ラベリング）

書籍 第3章「系列ラベリング問題の解き方」（3-1 固有表現認識、3-2 形態素解析、練習問題 1・2）に対応する演習ノートブックです。
CPU のみ・十数秒で実行でき、以下の内容を体験します。

1. `SudachiPy` による形態素解析と分割単位（短単位・中単位・長単位）の違い（3-2）
2. `SentencePiece` によるサブワード分割と未知語への振る舞い（3-2）
3. BIO（IOB2）タグによる固有表現のスパン表現（3-1、図 3.2）
4. CRF と素性テンプレート（ユニグラム・バイグラム素性）による固有表現認識の比較実験（3-2、練習問題 2）

## 1. 準備

形態素解析器 `SudachiPy`（辞書 `sudachidict_core`）、サブワード分割器 `SentencePiece`、および CRF ライブラリ `sklearn-crfsuite` をインストールします。

In [1]:
!pip install -q sudachipy sudachidict_core sentencepiece sklearn-crfsuite

In [2]:
import tempfile
from pathlib import Path

import pandas as pd
import sentencepiece as spm
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from sudachipy import dictionary, tokenizer

---

## 2. 形態素解析と分割単位（3-2「単語分割の基準」）

日本語は英語と異なり単語の境界が空白で区切られていないため、タスクや目的に応じて「どこまでを 1 単語とするか（分割単位）」を決める必要があります。
`SudachiPy` では、1 つの辞書で以下の 3 つの分割モードを切り替えられます。

- **Mode A（短単位相当）**: 最も細かい分割（検索や詳細な言語学的分析向け）
- **Mode B（中単位相当）**: 複合語の一部をまとめた中間的な分割
- **Mode C（長単位相当）**: 固有名詞や複合語をひとまとまりとして扱う分割（固有表現認識や情報抽出向け）

In [3]:
sudachi = dictionary.Dictionary().create()

sample_texts = [
    "東京都立大学で自然言語処理を学ぶ。",
    "関西国際空港から新型コロナウイルス感染症の対策会議へ向かう。",
]

mode_map = {
    "Mode A (短単位)": tokenizer.Tokenizer.SplitMode.A,
    "Mode B (中単位)": tokenizer.Tokenizer.SplitMode.B,
    "Mode C (長単位)": tokenizer.Tokenizer.SplitMode.C,
}

for text in sample_texts:
    print(f"入力文: {text}")
    for name, mode in mode_map.items():
        tokens = [m.surface() for m in sudachi.tokenize(text, mode)]
        print(f"  {name}: {' / '.join(tokens)}")
    print()

入力文: 東京都立大学で自然言語処理を学ぶ。
  Mode A (短単位): 東京 / 都立 / 大学 / で / 自然 / 言語 / 処理 / を / 学ぶ / 。
  Mode B (中単位): 東京 / 都立 / 大学 / で / 自然 / 言語 / 処理 / を / 学ぶ / 。
  Mode C (長単位): 東京都立大学 / で / 自然言語処理 / を / 学ぶ / 。

入力文: 関西国際空港から新型コロナウイルス感染症の対策会議へ向かう。
  Mode A (短単位): 関西 / 国際 / 空港 / から / 新型 / コロナ / ウイルス / 感染 / 症 / の / 対策 / 会議 / へ / 向かう / 。
  Mode B (中単位): 関西 / 国際 / 空港 / から / 新型 / コロナ / ウイルス / 感染症 / の / 対策 / 会議 / へ / 向かう / 。
  Mode C (長単位): 関西国際空港 / から / 新型コロナウイルス感染症 / の / 対策 / 会議 / へ / 向かう / 。



/tmp/ipykernel_897/475345972.py:1: DeprecationWarning: Dictionary.create() is deprecated, use Dictionary.tokenizer() instead
  sudachi = dictionary.Dictionary().create()


### 品詞情報・見出し語化（正規化）の確認

形態素解析では単語分割だけでなく、品詞（POS）や辞書形（終止形）・正規化形（表記揺れの統一）も同時に得られます。

In [4]:
text = "走ったり泳いだりして、コーヒーを飲みました。"
rows = []
for m in sudachi.tokenize(text, tokenizer.Tokenizer.SplitMode.C):
    rows.append({
        "表層形": m.surface(),
        "辞書形(見出し語)": m.dictionary_form(),
        "正規化形": m.normalized_form(),
        "品詞(大分類-中分類)": "-".join(m.part_of_speech()[:2]),
        "読み": m.reading_form(),
    })
pd.DataFrame(rows)

,表層形,辞書形(見出し語),正規化形,品詞(大分類-中分類),読み
0,走っ,走る,走る,動詞-一般,ハシッ
1,たり,たり,たり,助詞-副助詞,タリ
2,泳い,泳ぐ,泳ぐ,動詞-一般,オヨイ
3,だり,だり,たり,助詞-副助詞,ダリ
4,し,する,為る,動詞-非自立可能,シ
5,て,て,て,助詞-接続助詞,テ
6,、,、,、,補助記号-読点,、
7,コーヒー,コーヒー,コーヒー,名詞-普通名詞,コーヒー
8,を,を,を,助詞-格助詞,ヲ
9,飲み,飲む,飲む,動詞-一般,ノミ


---

## 3. 未知語とサブワード分割（3-2「サブワード化」）

辞書ベースの漢字仮名交じり形態素解析では、辞書に載っていない新語・固有名詞が細かくバラバラに切れたり未知語扱いになったりすることがあります。
一方、ニューラルモデル（BERT や機械翻訳など）では、コーパスから統計的に部分語（サブワード）の語彙を学習する **SentencePiece**（BPE やユニグラム言語モデル）がよく使われます。

小さなコーパスから SentencePiece モデルを学習し、学習データに出現する語と出現しない新語がどう分割されるかを確認してみましょう。

In [5]:
mini_corpus = [
    "自然言語処理の教科書を読みました。",
    "形態素解析は単語分割と品詞推定を行います。",
    "固有表現認識では人名や組織名を抽出します。",
    "東京都立大学の研究室で自然言語処理を研究しています。",
    "機械翻訳と文書要約は系列変換タスクです。",
    "深層学習と事前学習モデルが広く使われています。",
] * 15  # 頻度確保のため複製

with tempfile.TemporaryDirectory() as tmpdir:
    corpus_path = Path(tmpdir) / "corpus.txt"
    model_prefix = str(Path(tmpdir) / "sp_model")
    corpus_path.write_text("\n".join(mini_corpus), encoding="utf-8")

    spm.SentencePieceTrainer.train(
        input=str(corpus_path),
        model_prefix=model_prefix,
        vocab_size=360,
        model_type="unigram",
        character_coverage=1.0,
        byte_fallback=True,
        hard_vocab_limit=False,
        minloglevel=2,
        bos_id=-1,
        eos_id=-1,
    )
    sp = spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")

    for s in [
        "自然言語処理の研究をしています。",
        "ChatGPTで未知語を解析する。",  # 学習コーパスにないアルファベット・漢字列を含む文
    ]:
        print("入力:", s)
        print("  SentencePiece:", sp.encode_as_pieces(s))

入力: 自然言語処理の研究をしています。
  SentencePiece: ['▁', '自', '然', '言', '語', '処', '理', 'の', '研究', 'を', 'し', 'て', 'い', 'ま', 'す', '。']
入力: ChatGPTで未知語を解析する。
  SentencePiece: ['▁', '<0x43>', '<0x68>', '<0x61>', '<0x74>', '<0x47>', '<0x50>', '<0x54>', 'で', '<0xE6>', '<0x9C>', '<0xAA>', '<0xE7>', '<0x9F>', '<0xA5>', '語', 'を', '解', '析', 'す', '<0xE3>', '<0x82>', '<0x8B>', '。']


`ChatGPT` のような学習データにない単語でも、未知語トークン `<unk>` 1 つに潰れるのではなく、文字単位や既知の部分列にフォールバックして表現できることがわかります（`▁` は単語の先頭または空白を表す記号です）。

---

## 4. 固有表現認識と BIO タグ（3-1、図 3.2）

固有表現認識（NER）は、テキスト中の人名（`PER`）・組織名（`ORG`）・地名（`LOC`）などのスパンを **BIO（IOB2）タグ** による系列ラベリング問題として定式化します。

- `B-X`: 固有表現 `X` の先頭（Begin）
- `I-X`: 固有表現 `X` の内部（Inside）
- `O`: 固有表現以外（Outside）

`B-` と `I-` を区別することで、「東京（`B-LOC`）」「大阪（`B-LOC`）」のように同種の固有表現が連続しても 2 つの別々の固有表現として切り出すことができます。

In [6]:
def decode_bio(tokens, tags):
    '''トークン列と BIO タグ列から固有表現スパンを抽出する'''
    entities = []
    cur_type = None
    cur_tokens = []
    for tok, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if cur_tokens:
                entities.append((cur_type, "".join(cur_tokens)))
            cur_type = tag[2:]
            cur_tokens = [tok]
        elif tag.startswith("I-") and cur_type == tag[2:]:
            cur_tokens.append(tok)
        else:
            if cur_tokens:
                entities.append((cur_type, "".join(cur_tokens)))
                cur_type = None
                cur_tokens = []
    if cur_tokens:
        entities.append((cur_type, "".join(cur_tokens)))
    return entities

sample_toks = ["山田", "さん", "は", "東京", "大阪", "間", "を", "移動", "し", "た"]
sample_tags = ["B-PER", "O", "O", "B-LOC", "B-LOC", "O", "O", "O", "O", "O"]
print("抽出された固有表現:", decode_bio(sample_toks, sample_tags))

抽出された固有表現: [('PER', '山田'), ('LOC', '東京'), ('LOC', '大阪')]


---

## 5. 素性テンプレートと CRF による比較実験（3-2、練習問題 2）

書籍 3-2（図 3.3）および練習問題 2 で解説されているとおり、CRF などの系列ラベリングモデルでは**素性テンプレート**を用いて各トークンの周辺の表層形・品詞・文字種・バイグラム素性などを動的に展開します。

ここでは `SudachiPy`（Mode A）で形態素解析した小さな固有表現データセットを用意し、素性テンプレートを段階的に増やしたときの性能変化を確認します。

In [7]:
# (文, [(表層文字列, ラベル), ...]) の形式で簡易データセットを定義
raw_train = [
    ("佐藤さんは東京都立大学で研究している。", [("佐藤", "PER"), ("東京都立大学", "ORG")]),
    ("鈴木さんは京都大学を卒業した。", [("鈴木", "PER"), ("京都大学", "ORG")]),
    ("高橋さんは大阪大学に着任した。", [("高橋", "PER"), ("大阪大学", "ORG")]),
    ("田中さんは名古屋大学で講義を行った。", [("田中", "PER"), ("名古屋大学", "ORG")]),
    ("伊藤さんは九州大学から東京へ出張した。", [("伊藤", "PER"), ("九州大学", "ORG"), ("東京", "LOC")]),
    ("渡辺さんは北海道大学を訪問した。", [("渡辺", "PER"), ("北海道大学", "ORG")]),
    ("山本さんは東北大学で学会に参加した。", [("山本", "PER"), ("東北大学", "ORG")]),
    ("中村さんは横浜国立大学に在籍している。", [("中村", "PER"), ("横浜国立大学", "ORG")]),
    ("小林さんは筑波大学と理化学研究所と共同研究する。", [("小林", "PER"), ("筑波大学", "ORG"), ("理化学研究所", "ORG")]),
    ("加藤さんは渋谷と新宿の間を歩いた。", [("加藤", "PER"), ("渋谷", "LOC"), ("新宿", "LOC")]),
    ("吉田さんは札幌から福岡へ移動した。", [("吉田", "PER"), ("札幌", "LOC"), ("福岡", "LOC")]),
    ("山田さんは仙台で会議に出席した。", [("山田", "PER"), ("仙台", "LOC")]),
    ("松本さんは広島と名古屋を視察した。", [("松本", "PER"), ("広島", "LOC"), ("名古屋", "LOC")]),
    ("井上さんは沖縄から東京に戻った。", [("井上", "PER"), ("沖縄", "LOC"), ("東京", "LOC")]),
    ("木村さんは神戸大学で講演した。", [("木村", "PER"), ("神戸大学", "ORG")]),
    ("林さんは一橋大学の研究会に参加した。", [("林", "PER"), ("一橋大学", "ORG")]),
]

# 訓練データには一度も登場しない人名・組織名・地名を含むテストデータ
raw_test = [
    ("斎藤さんは千葉大学で発表を行った。", [("斎藤", "PER"), ("千葉大学", "ORG")]),
    ("清水さんは金沢大学から京都へ移動した。", [("清水", "PER"), ("金沢大学", "ORG"), ("京都", "LOC")]),
    ("森さんは産業技術総合研究所を訪問した。", [("森", "PER"), ("産業技術総合研究所", "ORG")]),
    ("池田さんは長崎から横浜へ出張した。", [("池田", "PER"), ("長崎", "LOC"), ("横浜", "LOC")]),
    ("橋本さんは岡山大学で学会に参加した。", [("橋本", "PER"), ("岡山大学", "ORG")]),
    ("石川さんは熊本と鹿児島を視察した。", [("石川", "PER"), ("熊本", "LOC"), ("鹿児島", "LOC")]),
]

def align_bio(text, entities):
    '''SudachiPy のトークン列に文字オフセット基準で BIO タグを付与する'''
    spans = []
    for ent_text, ent_type in entities:
        start = text.find(ent_text)
        assert start != -1, f"{ent_text} not in {text}"
        spans.append((start, start + len(ent_text), ent_type))

    morphemes = list(sudachi.tokenize(text, tokenizer.Tokenizer.SplitMode.A))
    tokens = []
    tags = []
    for m in morphemes:
        b, e = m.begin(), m.end()
        tag = "O"
        for sb, se, etype in spans:
            if b >= sb and e <= se:
                tag = f"B-{etype}" if b == sb else f"I-{etype}"
                break
        tokens.append((m.surface(), m.part_of_speech()[0], m.part_of_speech()[1]))
        tags.append(tag)
    return tokens, tags

train_data = [align_bio(t, ents) for t, ents in raw_train]
test_data = [align_bio(t, ents) for t, ents in raw_test]

print("トークン化・BIO付与の例:", list(zip([x[0] for x in train_data[0][0]], train_data[0][1])))

トークン化・BIO付与の例: [('佐藤', 'B-PER'), ('さん', 'O'), ('は', 'O'), ('東京', 'B-ORG'), ('都立', 'I-ORG'), ('大学', 'I-ORG'), ('で', 'O'), ('研究', 'O'), ('し', 'O'), ('て', 'O'), ('いる', 'O'), ('。', 'O')]


In [8]:
def char_type(s):
    if s.isdigit():
        return "DIGIT"
    if all("ァ" <= ch <= "ヶ" or ch == "ー" for ch in s):
        return "KATA"
    if all("ぁ" <= ch <= "ん" for ch in s):
        return "HIRA"
    if all("\u4e00" <= ch <= "\u9fff" for ch in s):
        return "KANJI"
    return "MIXED"

def extract_features(sent, i, level):
    word, pos1, pos2 = sent[i]
    feats = {"bias": 1.0, "w": word}

    if level >= 2:
        feats.update({
            "pos1": pos1,
            "pos2": pos2,
            "ctype": char_type(word),
        })

    if level >= 3:
        if i > 0:
            pw, ppos1, _ = sent[i - 1]
            feats.update({
                "-1:w": pw,
                "-1:pos1": ppos1,
                "-1_0:w": f"{pw}/{word}",      # バイグラム素性
            })
        else:
            feats["BOS"] = True

        if i < len(sent) - 1:
            nw, npos1, _ = sent[i + 1]
            feats.update({
                "+1:w": nw,
                "+1:pos1": npos1,
                "0_+1:w": f"{word}/{nw}",      # バイグラム素性
            })
        else:
            feats["EOS"] = True

    return feats

results = []
models = {}
for level, desc in [
    (1, "1. 表層形のみ"),
    (2, "2. +品詞・文字種"),
    (3, "3. +前後文脈・バイグラム素性"),
]:
    X_train = [[extract_features(s, i, level) for i in range(len(s))] for s, _ in train_data]
    y_train = [tags for _, tags in train_data]
    X_test = [[extract_features(s, i, level) for i in range(len(s))] for s, _ in test_data]
    y_test = [tags for _, tags in test_data]

    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
    crf.fit(X_train, y_train)
    y_pred = crf.predict(X_test)
    models[level] = (crf, y_pred)

    ent_labels = ["B-PER", "B-ORG", "I-ORG", "B-LOC"]
    f1_macro = crf_metrics.flat_f1_score(y_test, y_pred, average="macro", labels=ent_labels, zero_division=0)
    acc = crf_metrics.flat_accuracy_score(y_test, y_pred)
    results.append({"素性テンプレート": desc, "トークン正解率": round(acc, 3), "固有表現 Macro-F1": round(f1_macro, 3)})

pd.DataFrame(results)

,素性テンプレート,トークン正解率,固有表現 Macro-F1
0,1. 表層形のみ,0.857,0.614
1,2. +品詞・文字種,1.000,1.000
2,3. +前後文脈・バイグラム素性,1.000,1.000


未知の人名（斎藤・清水・森…）や地名（長崎・横浜…）は「表層形のみ」では見つけられませんが、**品詞（固有名詞）や前後の文脈（「〜さんは」「〜から」「〜へ」）の素性テンプレート**を加えることで、未知の固有表現も正しく抽出できるようになります。

最後に、素性レベル 3 の予測例と、CRF が学習したタグ間の遷移重み（高い遷移・低い遷移）を確認してみましょう。

In [9]:
crf3, y_pred3 = models[3]
for (sent, gold), pred in zip(test_data, y_pred3):
    toks = [w for w, _, _ in sent]
    print("文:", "".join(toks))
    print("  正解:", decode_bio(toks, gold))
    print("  予測:", decode_bio(toks, pred))

print("\n学習された遷移重み（上位5件・下位5件）:")
trans = sorted(crf3.transition_features_.items(), key=lambda x: x[1], reverse=True)
for (src, dst), w in trans[:5] + trans[-5:]:
    print(f"  {src:6s} -> {dst:6s}: {w:+.3f}")

文: 斎藤さんは千葉大学で発表を行った。
  正解: [('PER', '斎藤'), ('ORG', '千葉大学')]
  予測: [('PER', '斎藤'), ('ORG', '千葉大学')]
文: 清水さんは金沢大学から京都へ移動した。
  正解: [('PER', '清水'), ('ORG', '金沢大学'), ('LOC', '京都')]
  予測: [('PER', '清水'), ('ORG', '金沢大学'), ('LOC', '京都')]
文: 森さんは産業技術総合研究所を訪問した。
  正解: [('PER', '森'), ('ORG', '産業技術総合研究所')]
  予測: [('PER', '森'), ('ORG', '産業技術総合研究所')]
文: 池田さんは長崎から横浜へ出張した。
  正解: [('PER', '池田'), ('LOC', '長崎'), ('LOC', '横浜')]
  予測: [('PER', '池田'), ('LOC', '長崎'), ('LOC', '横浜')]
文: 橋本さんは岡山大学で学会に参加した。
  正解: [('PER', '橋本'), ('ORG', '岡山大学')]
  予測: [('PER', '橋本'), ('ORG', '岡山大学')]
文: 石川さんは熊本と鹿児島を視察した。
  正解: [('PER', '石川'), ('LOC', '熊本'), ('LOC', '鹿児島')]
  予測: [('PER', '石川'), ('LOC', '熊本'), ('LOC', '鹿児島')]

学習された遷移重み（上位5件・下位5件）:
  B-ORG  -> I-ORG : +2.033
  O      -> O     : +1.913
  I-ORG  -> I-ORG : +0.923
  B-PER  -> O     : +0.727
  O      -> B-LOC : +0.477
  B-PER  -> I-ORG : -0.095
  B-LOC  -> I-ORG : -0.637
  O      -> B-PER : -0.845
  B-ORG  -> O     : -1.141
  O      -> I-ORG : -1.362
